In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import os

os.makedirs('figures', exist_ok=True)

CSV = 'data/baselines_results.csv'
df = pd.read_csv(CSV, sep='\t')

SIZES = sorted(df['size'].unique())
STRATEGIES = sorted(df['strategy'].unique())
EVAL_COLS = [f'eval_{i:02d}' for i in range(1, 15)]

STRATEGY_NAMES = {
    'mpra_oracle':              'Malinois training sequences + oracle labels',
    'mpra_real':                'Malinois training sequences + real MPRA labels',
    'synth_oracle':             'Random synthetic + oracle labels',
    'dhs_topic':                'DHS topic-weighted',
    'dhs_random':               'DHS random',
    'sei_class':                'SEI class-balanced',
    'sei_random':               'SEI random',
    'dhs_sei':                  '½ DHS + ½ SEI',
    'dhs_synth':                '½ DHS + ½ synth',
    'sei_synth':                '½ SEI + ½ synth',
    'dhs_sei_synth':            '⅓ DHS + ⅓ SEI + ⅓ synthetic',
    'dhs_stratified':           'DHS component-stratified',
    'dhs_stratified_sei':       'DHS stratified + ½ SEI',
    'dhs_stratified_sei_synth': 'DHS stratified + ⅓ SEI + ⅓ synthetic',
}

# Mean and std across seeds
mean_df = df.groupby(['strategy', 'size'])[EVAL_COLS].mean().reset_index()
std_df  = df.groupby(['strategy', 'size'])[EVAL_COLS].std().reset_index()

# Best designed strategy on eval_01 at 50k (excluding reference strategies)
REFERENCE = {'mpra_oracle', 'mpra_real', 'synth_oracle'}
best_50k = mean_df[mean_df['size'] == 50000].set_index('strategy')['eval_01']
BEST_STRATEGY = best_50k.drop(index=[s for s in REFERENCE if s in best_50k.index]).idxmax()
print(f'Best strategy at 50k: {BEST_STRATEGY} ({best_50k[BEST_STRATEGY]:.4f})')
print(f'Random floor (synth_oracle): {best_50k["synth_oracle"]:.4f}')

In [ ]:
# ── Fig 2: Library composition drives performance — strip plot + rank heatmap ──
# Left:  per-seed eval_01 at 50k, strategies sorted by mean (horizontal beeswarm)
# Right: same row order, all 14 eval sets (rank consistency heatmap)

at_50k_all  = df[df['size'] == 50000].copy()
at_50k_mean = at_50k_all.groupby('strategy')['eval_01'].mean().sort_values(ascending=True)
strategy_order = list(at_50k_mean.index)   # worst → best (bottom → top)
y_pos  = {s: i for i, s in enumerate(strategy_order)}
n_strat = len(strategy_order)
friendly = [STRATEGY_NAMES.get(s, s) for s in strategy_order]

rng2 = np.random.RandomState(7)
JITTER = 0.22

fig = plt.figure(figsize=(13, 5.5), dpi=300)
gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 2.2], wspace=0.06)
ax_strip = fig.add_subplot(gs[0])
ax_heat  = fig.add_subplot(gs[1])

# Left: horizontal strip plot
for strat in strategy_order:
    rows = at_50k_all[at_50k_all['strategy'] == strat]['eval_01'].values
    y = y_pos[strat]
    jy = rng2.uniform(-JITTER, JITTER, len(rows))
    ax_strip.scatter(rows, y + jy, s=30, color='#4292c6', alpha=0.75,
                     linewidths=0, zorder=3)
    ax_strip.scatter(rows.mean(), y, s=60, color='#08306b',
                     linewidths=0, zorder=4, marker='|')

best_mean  = at_50k_mean.iloc[-1]
worst_mean = at_50k_mean.iloc[0]
ax_strip.annotate('', xy=(best_mean, n_strat - 0.45),
                  xytext=(worst_mean, n_strat - 0.45),
                  arrowprops=dict(arrowstyle='<->', color='#333333', lw=1.3))
ax_strip.text((best_mean + worst_mean) / 2, n_strat - 0.1,
              f'Δ = {best_mean - worst_mean:.3f}', ha='center', va='bottom',
              fontsize=8.5, color='#333333')

ax_strip.set_yticks(range(n_strat))
ax_strip.set_yticklabels(friendly, fontsize=7.5)
ax_strip.set_xlabel('Pearson r vs ground truth (eval_01)', fontsize=9)
ax_strip.set_xlim(0.55, 0.80)
ax_strip.set_ylim(-0.7, n_strat + 0.1)
ax_strip.set_title('50k sequences, 5 seeds each', fontsize=9, pad=6)
ax_strip.xaxis.grid(True, color='#eeeeee', zorder=0)
ax_strip.set_axisbelow(True)
ax_strip.spines[['top', 'right']].set_visible(False)
ax_strip.tick_params(left=False)

# Right: rank consistency heatmap
heat_data = at_50k_all.groupby('strategy')[EVAL_COLS].mean().loc[strategy_order]

im = ax_heat.imshow(heat_data.values, aspect='auto', cmap='Blues',
                    vmin=0.55, vmax=0.82, origin='lower')
ax_heat.set_xticks(range(len(EVAL_COLS)))
ax_heat.set_xticklabels(EVAL_COLS, rotation=45, ha='right', fontsize=7.5)
ax_heat.set_yticks(range(n_strat))
ax_heat.set_yticklabels([], fontsize=8)
ax_heat.tick_params(left=False)
ax_heat.set_xlabel('Eval set (anonymized)', fontsize=9)
ax_heat.set_title('Performance across all 14 eval sets (mean of 5 seeds)', fontsize=9, pad=6)
plt.colorbar(im, ax=ax_heat, shrink=0.85, label='Pearson r')

plt.tight_layout()
plt.savefig('figures/fig2_composition_drives_quality.pdf', bbox_inches='tight')
plt.savefig('figures/fig2_composition_drives_quality.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figures/fig2_composition_drives_quality.{pdf,png}')

In [ ]:
# ── Fig S1: Learning curves for all 14 strategies on primary eval ──
# Line = mean across 5 seeds. Shaded band = ±1 SD across 5 seeds.

import matplotlib.cm as cm

n_strats = len(STRATEGIES)
ncols = 4
nrows = int(np.ceil(n_strats / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3))
axes = axes.flatten()

colors = cm.tab20(np.linspace(0, 1, n_strats))
xtick_vals = [t / 1000 for t in SIZES]
fmt = mticker.FuncFormatter(lambda x, _: f'{int(x)}k')

for i, strat in enumerate(STRATEGIES):
    ax = axes[i]
    m = mean_df[mean_df['strategy'] == strat].sort_values('size')
    s = std_df[std_df['strategy']   == strat].sort_values('size')
    ax.plot(m['size'] / 1000, m['eval_01'], '-o', color=colors[i], lw=2, ms=4)
    ax.fill_between(m['size'] / 1000,
                    m['eval_01'] - s['eval_01'],
                    m['eval_01'] + s['eval_01'],
                    color=colors[i], alpha=0.2)
    ax.set_title(STRATEGY_NAMES.get(strat, strat), fontsize=8.5)
    ax.set_xscale('log')
    ax.set_xticks(xtick_vals)
    ax.xaxis.set_major_formatter(fmt)
    ax.tick_params(axis='x', labelsize=7, rotation=45)
    ax.set_xlabel('Library size', fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Pearson r', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Library size vs. surrogate model performance — 14 human-designed strategies',
             y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig('figures/figS1_all_learning_curves.pdf', bbox_inches='tight')
plt.savefig('figures/figS1_all_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/figS1_all_learning_curves.{pdf,png}')